In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import src

In [2]:
enoe = pd.read_parquet("outputs/enoe_workers.parquet")
od = pd.read_parquet("outputs/od_workers.parquet")

In [3]:
enoe_harmonized = src.harmonize_enoe_dataframe(enoe)
od_harmonized = src.harmonize_od_dataframe(od)

In [4]:
common_columns = ["genero", "ocupacion", "edad_num", "edad_cat", "escolaridad", "municipio", "estado_civil", "parentesco", "tamano_viv_num", "tamano_viv_cat", "sector", "sector_desconocido"]

assert enoe_harmonized.shape[0] == enoe.shape[0], "ENOE row count changed during harmonization."
assert od_harmonized.shape[0] == od.shape[0], "OD row count changed during harmonization."
assert set(common_columns).issubset(enoe_harmonized.columns), "Missing harmonized columns in ENOE."
assert set(common_columns).issubset(od_harmonized.columns), "Missing harmonized columns in OD."
assert "informal" in enoe_harmonized.columns, "Missing informal label in ENOE."

print("Harmonization validation completed successfully.")

Harmonization validation completed successfully.


In [5]:
for column in common_columns:
    print(f"\n{column}")
    print("ENOE:", enoe_harmonized[column].value_counts(dropna=False).to_dict())
    print("OD:", od_harmonized[column].value_counts(dropna=False).to_dict())


genero
ENOE: {'H': 3959, 'F': 2834}
OD: {'H': 16684, 'F': 10229}

ocupacion
ENOE: {'trabajador': 5391, 'independiente': 1235, 'otro': 167}
OD: {'trabajador': 23274, 'independiente': 3417, 'otro': 194, 'no_especificado': 28}

edad_num
ENOE: {27: 190, 24: 189, 26: 186, 22: 176, 29: 176, 50: 175, 23: 175, 31: 173, 30: 167, 32: 167, 33: 165, 28: 165, 38: 160, 36: 155, 40: 153, 35: 153, 37: 152, 41: 146, 46: 145, 39: 144, 34: 143, 45: 142, 20: 139, 42: 139, 43: 138, 53: 138, 44: 135, 21: 135, 49: 135, 25: 134, 54: 128, 52: 127, 51: 127, 48: 125, 19: 113, 47: 110, 55: 109, 18: 108, 56: 102, 57: 87, 17: 84, 59: 79, 61: 77, 58: 75, 60: 64, 62: 58, 16: 58, 63: 58, 64: 54, 65: 44, 66: 35, 67: 31, 68: 30, 15: 29, 69: 27, 70: 25, 71: 19, 14: 17, 75: 15, 13: 11, 72: 11, 74: 11, 73: 9, 76: 7, 80: 5, 77: 5, 12: 4, 78: 4, 82: 4, 79: 4, 81: 3, 98: 2, 85: 2, 84: 2, 92: 1, 88: 1, 87: 1, 94: 1}
OD: {31: 1158, 41: 1095, 26: 843, 51: 818, 46: 814, 33: 805, 39: 800, 29: 767, 36: 753, 30: 750, 43: 699, 28: 6

In [6]:
print("ENOE municipalities mapped as 'otro':")
print(enoe_harmonized.loc[enoe_harmonized["municipio"] == "otro", "mun"].value_counts(dropna=False))

print("\nOD municipalities mapped as 'otro':")
print(od_harmonized.loc[od_harmonized["municipio"] == "otro", "Municipio"].value_counts(dropna=False))

print("OD education mapped as 'no_especificado':")
print(od_harmonized.loc[od_harmonized["escolaridad"] == "no_especificado", "Escolaridad"].value_counts(dropna=False))

print("OD sectors mapped as 'no_especificado':")
print(od_harmonized.loc[od_harmonized["sector_desconocido"], "Giro de la empresa donde trabaja:"].value_counts(dropna=False))

ENOE municipalities mapped as 'otro':
mun
23     148
123    142
93     120
67     118
18     101
82      95
6       80
53      62
37      60
50      55
35      53
80      51
100     43
36      39
73      36
10      36
81      36
22      33
47      33
1       32
66      32
63      32
13      27
108     27
45      25
119     24
74      24
27      22
68      17
58      15
30      13
Name: count, dtype: Int64

OD municipalities mapped as 'otro':
Series([], Name: count, dtype: Int64)
OD education mapped as 'no_especificado':
Escolaridad
<NA>       5037
No sabe     158
Name: count, dtype: Int64
OD sectors mapped as 'no_especificado':
Giro de la empresa donde trabaja:
<NA>    9484
Name: count, dtype: Int64


In [7]:
print(f"ENOE workers: {len(enoe_harmonized):,}")
print(f"OD workers: {len(od_harmonized):,}")
print(f"ENOE unknown sectors: {enoe_harmonized['sector_desconocido'].sum():,}")
print(f"OD unknown sectors: {od_harmonized['sector_desconocido'].sum():,}")

ENOE workers: 6,793
OD workers: 26,913
ENOE unknown sectors: 37
OD unknown sectors: 9,484


In [8]:
output_directory = Path("outputs")
output_directory.mkdir(exist_ok=True)

enoe_harmonized.to_parquet(output_directory / "enoe_harmonized.parquet", index=False)
od_harmonized.to_parquet(output_directory / "od_harmonized.parquet", index=False)